# 04 · B2 — QLoRA sơ bộ (1 epoch) Qwen2.5-VL-3B trên ViVQA, GPU L4 (W7)

**Vấn đề:** B1 (zero-shot, notebook 02) đạt ViVQA EM 0,2209 · ANLS 0,2861. Phần lớn lỗi là **định dạng câu trả lời**: model trả lời dài ("Trung tâm Y tế Huyện Cao Lộc."), chen tiếng Anh ("truck"), dùng từ khác gold ("trong nhà hàng" thay vì "phòng bếp"). QLoRA học đúng kiểu đáp án ngắn của dataset nên có thể sửa nhóm lỗi này — notebook đo xem sửa được bao nhiêu.

**B2 khác B1 đúng một biến: có adapter LoRA hay không.** Mọi thứ khác giữ nguyên:

| Điều kiện | B1 (mốc) | B2 (notebook này) |
|---|---|---|
| Model gốc | Qwen2.5-VL-3B-Instruct, 4-bit NF4, bf16, 512 token ảnh | **y hệt** |
| Prompt, decoding | P0 zero-shot, greedy, `max_new_tokens`=32 | **y hệt** |
| Test | ViVQA test 3.001 câu (khoá từ commit 37da80a) | **y hệt** |
| GPU | L4 | **y hệt** |
| Adapter | không | **LoRA r=16 trên q, k, v, o** (7.372.800 tham số = 0,196%) |

**Huấn luyện (chốt trước khi chạy):** ViVQA train 10.799 mẫu **giữ nguyên, chưa làm sạch nhiễu** (làm sạch là ablation riêng, để không trộn hai biến) · **1 epoch = 675 step** · lr 2e-4, batch hiệu dụng 16, warmup 3% · seed 42. Đây là run **sơ bộ cho Review 2**; run chính thức (3 epoch, chọn epoch trên dev) để W8.

**Luật đánh giá (chốt trước khi xem kết quả):** B2 được coi là **cải thiện** so với B1 nếu EM trên ViVQA test tăng **và** kiểm định McNemar chính xác cho p < 0,05. ANLS báo cáo kèm CI95 bootstrap ghép cặp. Kết quả được ghi lại **dù tốt hay xấu**.

**Thời gian ước tính:** train ≈3 giờ (smoke W7: ≈2,94 giờ/epoch) + đánh giá test ≈30 phút → **≈3,5 giờ, ≈6 CU**. Giữ máy bật, tắt Sleep, không đóng tab.

**Nếu Colab rớt:** chạy lại từ phần 1 rồi **Run all** — train tự chạy tiếp từ checkpoint gần nhất (lưu mỗi 200 step, mất tối đa ≈50 phút), đánh giá tự bỏ qua các lô đã xong.

## 1. Thiết lập môi trường

Giống hệt notebook 02 (cùng thư viện, cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (≥ commit thêm `--adapter`/`--epochs`), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ B2 phải chạy trên L4 (cùng GPU với B1). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

Kiểm trong vài giây, trước khi nạp model: config đúng bản chốt W07, có `train.json`, có kết quả B1 để so, và selftest của hai script vẫn đạt trên đúng commit này.

*Kỳ vọng: `bfloat16 | 401408 | nf4 | bfloat16`, `train: 10799 mẫu`, `B1 test: 3001 dự đoán`, hai dòng `SELFTEST OK`.*

In [ ]:
import json, yaml

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
CHUNK      = 500
EVAL_CROSS_VITEXTVQA = False   # True: đánh giá thêm adapter ViVQA trên ViTextVQA 2.000 câu (~30 phút, xem có "chuyển giao" không)

RUN_TRAIN = 'W07_B2_qlora_r16_qwen25vl_vivqa_ep1_bf16_L4_seed42'
ADAPTER   = f'{DRIVE_ROOT}/adapters/{RUN_TRAIN}/epoch_1'
RUN_EVAL  = 'W07_B2_eval_qlora_r16_ep1_vivqa_test_bf16_L4_seed42'
B1_RUN    = 'W07_B1_zeroshot_qwen25vl_vivqa_test_bf16_L4_seed42'
RUN_CROSS = 'W07_B2_eval_qlora_r16_ep1_vivqa_on_vitextvqa_test2000_bf16_L4_seed42'
B1_CROSS  = 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42'

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
mdl, q, t = cfg['model'], cfg['quantization'], cfg['training']
print('torch_dtype | max_pixels | quant | compute dtype')
print(f"{mdl['torch_dtype']} | {mdl['max_pixels']} | {q.get('bnb_4bit_quant_type')} | {q.get('bnb_4bit_compute_dtype')}")
assert (mdl['torch_dtype'], mdl['max_pixels'], q.get('bnb_4bit_quant_type'), q.get('bnb_4bit_compute_dtype')) \
       == ('bfloat16', 401408, 'nf4', 'bfloat16'), 'Config KHÔNG phải bản chốt W07 -> dừng, báo Claude'
print(f"LoRA r={cfg['lora']['r']} alpha={cfg['lora']['lora_alpha']} on {cfg['lora']['target_modules']} | "
      f"lr={t['learning_rate']} | batch hiệu dụng={t['per_device_batch_size'] * t['gradient_accumulation_steps']}")

train = json.load(open(f'{DRIVE_ROOT}/data/vivqa/train.json', encoding='utf-8'))
print('train:', len(train), 'mẫu')
b1 = json.load(open(f'{EXP_ROOT}/{B1_RUN}/predictions.json', encoding='utf-8'))
print('B1 test:', len(b1), 'dự đoán')
assert len(b1) == 3001, 'Thiếu kết quả B1 trên Drive -> chạy notebook 02 trước'

!python -m src.adaptation.train_qlora --selftest 2>&1 | tail -n 1
!python -m src.eval.run_baseline --selftest 2>&1 | tail -n 1

## 3. Train QLoRA 1 epoch (≈3 giờ)

Chạy bằng `!python` (tiến trình riêng) để VRAM được trả lại sạch trước bước đánh giá. Đã có `train_log.json` thì bỏ qua.

*Kỳ vọng: `trainable 7,372,800 / 3,761,995,776 (0.196%) -> PEFT-only OK`, `lịch: 675 step tổng`, loss in mỗi 10 step, cuối cùng `💾 adapter epoch 1` và `== XONG`. Nếu bị ngắt: chạy lại cell này, phải thấy `↻ CHẠY TIẾP từ checkpoint-…`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{RUN_TRAIN}/train_log.json'):
    print('⏭ đã train xong -> bỏ qua')
else:
    !python -u -m src.adaptation.train_qlora --dataset vivqa --run_name {RUN_TRAIN} --epochs 1

## 4. Kiểm tra đường train

Loss phải là số hữu hạn và có xu hướng giảm; `final_loss` là trung bình loss của phiên (đã hiệu chỉnh nếu có resume).

*Kỳ vọng: `steps` = 675, `trainable_pct` = 0,196, loss cuối thấp hơn rõ loss đầu, có thư mục `epoch_1` chứa `adapter_model.safetensors`.*

In [ ]:
log = json.load(open(f'{EXP_ROOT}/{RUN_TRAIN}/train_log.json', encoding='utf-8'))
m = log['meta']
print(f"global_step {m['cost']['global_step']} | {m['cost']['sec_per_step']} s/step | VRAM đỉnh {m['cost']['peak_vram_gb']} GB")
print(f"final_loss {m['cost']['final_loss']} | last_logged_loss {m['cost'].get('last_logged_loss')} | resumed_from {m['resumed_from']}")
print(f"trainable {m['trainable_params']:,} / {m['total_params']:,} = {m['trainable_pct']}%  (PEFT-only)")
losses = [(h['step'], h['loss']) for h in log['log_history'] if 'loss' in h]
for s, l in losses[::max(1, len(losses) // 12)] + losses[-1:]:
    print(f'  step {s:>4} | loss {float(l):.4f}')
print('adapter:', sorted(os.listdir(ADAPTER)))

## 5. Đánh giá B2 trên đúng tập test của B1 (≈30 phút)

Cùng script `run_baseline.py`, cùng config, chỉ thêm `--adapter`. Script ghi sha256 của adapter vào `run_signature.json`, nên không thể vô tình chấm nhầm adapter khác dưới cùng tên run.

*Kỳ vọng: dòng `adapter: … (≈29 MB, sha256 …)`, `đã gắn adapter LoRA`, 7 lô, cuối cùng `== KẾT QUẢ: EM=…`.*

In [ ]:
EVALS = [('vivqa', RUN_EVAL)] + ([('vitextvqa_official', RUN_CROSS)] if EVAL_CROSS_VITEXTVQA else [])
for ds, run in EVALS:
    if os.path.exists(f'{EXP_ROOT}/{run}/metrics.json'):
        print(f'⏭ {run}: đã có metrics.json -> bỏ qua'); continue
    print(f'\n▶ {run}')
    !python -u -m src.eval.run_baseline --dataset {ds} --run_name {run} --chunk {CHUNK} --adapter {ADAPTER}

## 6. B1 vs B2 — so sánh ghép cặp

Ba bảng: (1) điểm tổng + kiểm định McNemar + CI95 ANLS (script `compare_runs.py` có sẵn), (2) theo loại câu hỏi, (3) độ dài câu trả lời — kiểm tra trực tiếp giả thuyết "QLoRA sửa định dạng đáp án".

*Kỳ vọng: dòng `KẾT LUẬN` theo luật chốt ở đầu notebook.*

In [ ]:
import re
import pandas as pd
from src.eval.metrics import exact_match, normalize_vi

def load(run):
    d = f'{EXP_ROOT}/{run}'
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')),
            json.load(open(f'{d}/predictions.json', encoding='utf-8')))

def is_degenerate(p):
    return not re.sub(r'[!\s]', '', p or '')

PAIRS = [('vivqa', B1_RUN, RUN_EVAL)] + ([('vitextvqa (chuyển giao)', B1_CROSS, RUN_CROSS)] if EVAL_CROSS_VITEXTVQA else [])
CMP, rows = {}, []
for name, ra, rb in PAIRS:
    out = f'{EXP_ROOT}/W07_B2_summary_L4_seed42/paired_B1_vs_B2_{name.split()[0]}.json'
    !python src/eval/compare_runs.py --run_a {EXP_ROOT}/{ra} --run_b {EXP_ROOT}/{rb} --label_a B1 --label_b B2 --out {out}
    c = json.load(open(out, encoding='utf-8')); CMP[name] = c
    (ma, pa), (mb, pb) = load(ra), load(rb)
    for lab, mm, pp in [('B1 zero-shot', ma, pa), ('B2 QLoRA 1 epoch', mb, pb)]:
        rows.append({'tập': name, 'run': lab, 'EM': mm['metrics']['exact_match'], 'VQA-Acc': mm['metrics']['vqa_accuracy'],
                     'ANLS': mm['metrics']['anls'], 'token-F1 (phụ)': mm['metrics'].get('token_f1'),
                     'câu hỏng': sum(is_degenerate(p['prediction']) for p in pp), 's/câu': mm['meta']['sec_per_sample']})
print('==== Bảng 1. B1 vs B2 ====')
display(pd.DataFrame(rows).round(4))
for name, c in CMP.items():
    verdict = ('CẢI THIỆN (EM tăng, p < 0,05)' if c['em_diff'] > 0 and c['p_value'] < 0.05 else
               'KÉM HƠN (EM giảm, p < 0,05)' if c['em_diff'] < 0 and c['p_value'] < 0.05 else
               'CHƯA KẾT LUẬN ĐƯỢC (p ≥ 0,05)')
    print(f"{name}: chỉ B1 đúng {c['b']} | chỉ B2 đúng {c['c']} | McNemar p = {c['p_value']:.2g} | "
          f"ΔEM {c['em_diff']:+.4f} | ΔANLS {c['anls_diff']:+.4f} CI95 [{c['anls_ci95'][0]:+.4f}; {c['anls_ci95'][1]:+.4f}]")
    print(f'  ➜ KẾT LUẬN ({name}): {verdict}')

(ma, pa), (mb, pb) = load(B1_RUN), load(RUN_EVAL)
types = ['color', 'location', 'object', 'number']
tt = [{'loại': ty, 'n': mb['meta'].get('question_type_counts', {}).get(ty),
       'B1 EM': ma['metrics'].get(f'exact_match__{ty}'), 'B2 EM': mb['metrics'].get(f'exact_match__{ty}'),
       'B1 ANLS': ma['metrics'].get(f'anls__{ty}'), 'B2 ANLS': mb['metrics'].get(f'anls__{ty}')} for ty in types]
print('\n==== Bảng 2. ViVQA theo loại câu hỏi ====')
display(pd.DataFrame(tt).round(4))

nw = lambda s: len(normalize_vi(s).split())
L = {'gold (đáp án đầu)': [nw(p['answers'][0]) for p in pa], 'B1': [nw(p['prediction']) for p in pa],
     'B2': [nw(p['prediction']) for p in pb]}
print('\n==== Bảng 3. Độ dài câu trả lời (số từ sau chuẩn hoá) ====')
display(pd.DataFrame([{'nguồn': k, 'trung bình': sum(v) / len(v), '> 3 từ (%)': 100 * sum(x > 3 for x in v) / len(v)}
                      for k, v in L.items()]).round(2))

## 7. Soi các câu đổi kết quả

8 câu B2 sửa đúng và 8 câu B2 làm sai (seed 0). Đây là đầu vào cho error analysis + hallucination analysis ở W8.

*Kỳ vọng: nhóm "B2 sửa đúng" chủ yếu là câu B1 trả lời dài hoặc dùng từ khác gold.*

In [ ]:
import random
b2 = {p['question_id']: p for p in pb}
fixed  = [p for p in pa if not exact_match(p['prediction'], p['answers']) and exact_match(b2[p['question_id']]['prediction'], p['answers'])]
broken = [p for p in pa if exact_match(p['prediction'], p['answers']) and not exact_match(b2[p['question_id']]['prediction'], p['answers'])]
for title, group in [('B2 SỬA ĐÚNG', fixed), ('B2 LÀM SAI', broken)]:
    print(f'\n######## {title}: {len(group)} câu ########')
    for p in random.Random(0).sample(group, min(8, len(group))):
        print(f"Q: {p['question']}\n   B1  : {p['prediction']!r}\n   B2  : {b2[p['question_id']]['prediction']!r}\n   GOLD: {p['answers'][:3]}")

## 8. Lưu tóm tắt B2

Ghi `experiments/W07_B2_summary_L4_seed42/b2_summary.json` (điều kiện, chi phí train, bảng 1–3, kết luận theo luật). Cursor chép file này + `train_log.json`, `run_signature.json` của run train + `metrics.json`, `predictions.json`, `run_signature.json` của run đánh giá + các file `paired_*.json` vào repo. **Không** chép `checkpoints/`, `chunks/`, adapter.

*Kỳ vọng: `Ghi -> …/b2_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
OUT_DIR = f'{EXP_ROOT}/W07_B2_summary_L4_seed42'
os.makedirs(OUT_DIR, exist_ok=True)
summary = {
    'what': 'B2 = QLoRA sơ bộ 1 epoch trên ViVQA train; so với B1 (zero-shot) cùng model 4-bit, prompt, decoding, test, GPU',
    'pre_registered_rule': 'cải thiện nếu EM tăng và McNemar exact p < 0.05 trên ViVQA test; ANLS kèm CI95 bootstrap ghép cặp',
    'train': {'run': RUN_TRAIN, 'adapter': ADAPTER, 'epochs': 1, 'global_step': m['cost']['global_step'],
              'sec_per_step': m['cost']['sec_per_step'], 'train_hours': round(m['cost']['train_runtime_sec'] / 3600, 2),
              'peak_vram_gb': m['cost']['peak_vram_gb'], 'final_loss': m['cost']['final_loss'],
              'resumed_from': m['resumed_from'], 'trainable_params': m['trainable_params'],
              'trainable_pct': m['trainable_pct'], 'data': 'ViVQA train 10.799 (chưa làm sạch nhiễu)'},
    'eval_runs': {name: rb for name, _, rb in PAIRS},
    'table1': rows, 'table2_by_type': tt,
    'table3_answer_length': {k: sum(v) / len(v) for k, v in L.items()},
    'paired': {k: {kk: vv for kk, vv in c.items() if kk != 'only_b_question_ids'} for k, c in CMP.items()},
    'gpu': GPU,
}
with open(f'{OUT_DIR}/b2_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/b2_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')